<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-05/week-05/week5_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 (starter): RAG Pipeline with Retrieval Evaluation
Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [1]:
import os, re, pathlib, requests
from bs4 import BeautifulSoup
import numpy as np

os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())

from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# TODO (you): replace with your own technical-doc corpus. This placeholder is one short doc.
DOC_URLS = {
    'docker_overview': 'https://docs.docker.com/get-started/docker-overview/',
    'what_is_container': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/',
    'what_is_image': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-an-image/',
    'what_is_registry': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-registry/',
    'what_is_compose': 'https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-docker-compose/',
    'image_layers': 'https://docs.docker.com/get-started/docker-concepts/building-images/understanding-image-layers/',
    'build_tag_publish': 'https://docs.docker.com/get-started/docker-concepts/building-images/build-tag-and-publish-an-image/',
    'running_containers': 'https://docs.docker.com/engine/containers/run/',
    'volumes': 'https://docs.docker.com/engine/storage/volumes/',
    'networking': 'https://docs.docker.com/engine/network/'
}

def fetch_doc(url):
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    for tag in soup(['script', 'style', 'nav', 'footer']):
        tag.decompose()

    # Added: preserve separate HTML text blocks so paragraph chunking works.
    blocks = []

    for element in soup.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li']):
        text = re.sub(r'\s+', ' ', element.get_text(' ', strip=True)).strip()

        if text:
            blocks.append(text)

    # Added: separate each text block with a blank line for chunk_paragraph().
    return '\n\n'.join(blocks)

DOCS = {name: fetch_doc(url) for name, url in DOC_URLS.items()}

for name, text in DOCS.items():
    print(f'{name:25s} -> {len(text):,} characters')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

docker_overview           -> 10,068 characters
what_is_container         -> 7,332 characters
what_is_image             -> 9,962 characters
what_is_registry          -> 6,598 characters
what_is_compose           -> 7,784 characters
image_layers              -> 10,494 characters
build_tag_publish         -> 8,301 characters
running_containers        -> 25,968 characters
volumes                   -> 18,730 characters
networking                -> 9,883 characters


In [2]:
def chunk_fixed(text, size, overlap):

    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0

    while i < len(text):

        out.append(text[i:i+size]); i += size - overlap

    return out

def chunk_paragraph(text):

    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]


# Added: apply each chunking configuration to all 10 documents.
configs = {
    'small (120/20)': [(name, chunk) for name, text in DOCS.items() for chunk in chunk_fixed(text, 120, 20)],
    'large (320/40)': [(name, chunk) for name, text in DOCS.items() for chunk in chunk_fixed(text, 320, 40)],
    'by-paragraph': [(name, chunk) for name, text in DOCS.items() for chunk in chunk_paragraph(text)]
}

for n, c in configs.items():

    print(f'{n:16s} -> {len(c)} chunks')


def build(chunks):

    # Added: embed only the text portion of each document chunk.
    e = embedder.encode([chunk for name, chunk in chunks], normalize_embeddings=True).astype('float32')

    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx


def retrieve(idx, chunks, q, k=3):

    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')

    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]

small (120/20)   -> 1148 chunks
large (320/40)   -> 414 chunks
by-paragraph     -> 938 chunks


## Part 1: Ingesting and Chunking the Corpus

For this RAG pipeline, I used 10 official Docker documentation pages as the document corpus. The documents were collected from their URLs and cleaned to remove webpage elements such as scripts, styles, navigation, and footers.

The starter code was slightly modified because it was designed around one placeholder document. I changed the document loading to support all 10 documents and preserved the document boundaries so each chunk could be traced back to its original source. I also preserved text block boundaries so the paragraph-based configuration would work correctly.

The corpus was then divided into three chunking configurations: small chunks (120 characters with 20-character overlap), large chunks (320 characters with 40-character overlap), and paragraph-based chunks. These configurations will be compared later using retrieval precision and recall.

## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. This is the model-free half of answer quality.

In [3]:
# TODO (you): your own queries and the answer fact each query must contain.
qa = [
    ('what is a Docker container', 'isolated processes'),
    ('what is a Docker image', 'read-only template'),
    ('what is a Docker registry', 'store and distribute images'),
    ('what is Docker Compose', 'multi-container applications'),
    ('what are Docker image layers', 'each layer'),
    ('how do you build tag and publish an image', 'docker push'),
    ('how do you run a Docker container', 'docker run'),
    ('what are Docker volumes used for', 'persist data'),
    ('what is Docker networking', 'network drivers'),
    ('what is Docker', 'open platform')
]

indices = {n: build(c) for n, c in configs.items()}

# TODO (you): compare at least two embedding models OR at least three chunking
# configurations on at least ten queries. Label relevant chunks in advance,
# report chunk-level precision and recall, and explain where the choices disagree.
# Calculate chunk-level precision and recall for each configuration.
for n, c in configs.items():

    retrieved = 0
    relevant_retrieved = 0
    relevant_total = 0

    for q, answer_fact in qa:

        results = retrieve(indices[n], c, q, k=3)

        # Count retrieved chunks and relevant retrieved chunks.
        retrieved += len(results)
        relevant_retrieved += sum(
            1 for doc_id, chunk in results
            if answer_fact.lower() in chunk.lower()
        )

        # Count all relevant chunks available in the ground truth.
        relevant_total += sum(
            1 for doc_id, chunk in c
            if answer_fact.lower() in chunk.lower()
        )

    # Precision@3 = relevant chunks retrieved / total chunks retrieved.
    precision = relevant_retrieved / retrieved

    # Recall@3 = relevant chunks retrieved / total relevant chunks available.
    recall = relevant_retrieved / relevant_total

    print(f'{n:16s} precision@3: {precision:.2f}  recall@3: {recall:.2f}')

# TODO (you): explain where the chunking choices disagree and why.

small (120/20)   precision@3: 0.13  recall@3: 0.06
large (320/40)   precision@3: 0.20  recall@3: 0.10
by-paragraph     precision@3: 0.13  recall@3: 0.07


### Part 2: Evaluating Chunking Choices
The large chunk configuration performed the best overall, with the highest precision and recall. I think this is because the larger chunks contain important context, making it more likely that the answer fact remains together in the retrieved chunk. The small chunks contain less information, so the answer can be split across multiple chunks and become harder to retrieve. The paragraph configuration was near identical to the small chunks, with nearly identical precision and only slightly higher recall.

Although the large chunks performed better, the overall precision and recall were still low. One possible reason is that the Docker documentation uses similar terminology across many of the documents. Terms such as containers, images, networks, and Docker appear throughout the corpus. It could make it harder for the embedding model to distinguish between closely related topics and retrieve relevant chunks. Since only the top three chunks were retrieved for each query, the system found some relevant information but still missed other relevant chunks. These results suggest that larger chunks preserve context, but chunk size alone does not solve the retrieval issues.


## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [4]:
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [5]:
def gemini_chat(messages, model='gemini-2.5-flash', **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(
        api_key=os.environ['GEMINI_API_KEY'],
        base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
    )
    return client.chat.completions.create(
        model=model,
        messages=messages,
        **kw
    ).choices[0].message.content

def answer(q, config='by-paragraph'):
    ctx = retrieve(indices[config], configs[config], q, k=3)
    prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(chunk for doc_id, chunk in ctx) + f'\nQuestion: {q}\nAnswer:'
    out = gemini_chat([{'role':'user','content':prompt}])
    return out if out is not None else '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'

print(answer('what is a Docker container'))

The provided context does not define what a Docker container is.


In [6]:
ctx = retrieve(indices['by-paragraph'], configs['by-paragraph'], 'what is a Docker container', k=3)

for doc_id, chunk in ctx:
    print(f'\n--- {doc_id} ---')
    print(chunk)


--- docker_overview ---
What is Docker?

--- docker_overview ---
What can I use Docker for?

--- what_is_container ---
Why Docker?


In [7]:
for config in configs:
    print(f'\n===== {config} =====')

    ctx = retrieve(
        indices[config],
        configs[config],
        'what is a Docker container',
        k=3
    )

    for doc_id, chunk in ctx:
        print(f'\n--- {doc_id} ---')
        print(chunk)


===== small (120/20) =====

--- running_containers ---
 started Guides Reference Running containers Docker runs processes in isolated containers. A container is a process whic

--- docker_overview ---
des Manuals Reference What is Docker? Docker is an open platform for developing, shipping, and running applications. Doc

--- docker_overview ---
hare containerized applications and microservices. Docker Desktop includes the Docker daemon ( dockerd ), the Docker cli

===== large (320/40) =====

--- docker_overview ---
When enabled, Gordon considers the current page you're viewing to provide more relevant answers. Guides Manuals Reference What is Docker? Docker is an open platform for developing, shipping, and running applications. Docker enables you to separate your applications from your infrastructure so you can deliver software q

--- docker_overview ---
res of the Linux kernel to deliver its functionality. Docker uses a technology called namespaces to provide the isolated workspace c

Use the gemini model to see if the large model can solve the question since it performs better

In [9]:
print(answer('what is a Docker container', config='large (320/40)'))

A Docker container is a loosely isolated environment where an application can be packaged and run. It is an isolated workspace provided by Docker using namespaces, and containers are lightweight, containing everything needed for an application.


### Part 3: Explanation
For the grounded generation step, I used the top three retrieved chunks as the context for Gemini. I first tested the by-paragraph configuration and found that the retrieved chunks did not have enough information to explain what a Docker container is. Gemini could not answer the question because it was instructed to only use the provided context.

I then compared the retrieved chunks from all three configurations without using Gemini calls. The large chunks retrieved multiple chunks that directly explained containers, while the small chunks also retrieved some relevant information. I used the large chunk configuration for the final generation, and Gemini was able to provide a grounded answer. This shows how the quality of retrieval directly affects the generated answer.


## Part 4 and 5: failure, submit
**TODO (you):** show a query where retrieval surfaces the wrong chunk or a chunk config splits a fact, and explain the mitigation. Then open a pull request with a result summary and review a classmate's Week 5 PR (the term's formal peer review). Use the rubric attached to the Canvas assignment for grading criteria and point values.

In [13]:
q = 'what is a Docker image'
answer_fact = 'read-only template'

for config in configs:
    matches = [
        (doc_id, chunk)
        for doc_id, chunk in configs[config]
        if answer_fact.lower() in chunk.lower()
    ]

    print(f'\n===== {config} =====')
    print(f'Relevant chunks containing "{answer_fact}": {len(matches)}')

    for doc_id, chunk in matches[:3]:
        print(f'\n--- {doc_id} ---')
        print(chunk)


===== small (120/20) =====
Relevant chunks containing "read-only template": 1

--- docker_overview ---
This section is a brief overview of some of those objects. Images An image is a read-only template with instructions for

===== large (320/40) =====
Relevant chunks containing "read-only template": 1

--- docker_overview ---
ief overview of some of those objects. Images An image is a read-only template with instructions for creating a Docker container. Often, an image is based on another image, with some additional customization. For example, you may build an image that is based on the Ubuntu image but includes the Apache web server and yo

===== by-paragraph =====
Relevant chunks containing "read-only template": 1

--- docker_overview ---
An image is a read-only template with instructions for creating a Docker container. Often, an image is based on another image, with some additional customization. For example, you may build an image that is based on the Ubuntu image but includes th

In [14]:
print(answer('what is a Docker image', config='small (120/20)'))

A Docker image contains instructions for creating a Docker container. Often, an image is based on another image, with some additional custom.


### Part 4: Failure Case Explanation

For my retrieval failure, I used the query "what is a Docker image" with the small chunk configuration. The correct chunk containing the fact that an image is a "read-only template" exists in the corpus, but it was not included in the top three retrieved chunks.

Instead, the results contained other information about Docker images, such as image tags and Docker Hub. When I passed these chunks to Gemini, it gave a related answer but did not include the key fact that an image is a "read-only template." This shows that the information was in the corpus, but the retrieval selected related chunks instead of the chunk with the answer.

One way I could improve this is by keeping metadata such as the document title and section heading with each chunk. This could give the embedding search more context about what each chunk is specifically about and help distinguish between similar topics across the Docker documentation.




----
**Note**: response to peer PR is linked in my PR